### Human-in-the-loop
Use the human-in-the-loop (HITL) flow to pause agent execution until a person approves or rejects sensitive tool calls.

[See the details](https://openai.github.io/openai-agents-python/human_in_the_loop/)

In [2]:
from agents import (Agent, 
                    Runner, AsyncOpenAI, 
                    OpenAIChatCompletionsModel, set_tracing_disabled)

from agents.decorators import tool

In [3]:
# Tracing disabled
set_tracing_disabled(disabled=True)

In [4]:
import os
gemini_api = os.environ['GEMINI_API_KEY']

In [5]:
# 1. Which LLM Service?
external_client = AsyncOpenAI(
    api_key=gemini_api,
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/",
)

In [6]:
# 2. Which LLM Model?
llm_model = OpenAIChatCompletionsModel(
    model="gemini-2.5-flash",
    openai_client=external_client
)

##### Step-1

In [13]:
# 1. A sensitive tool that needs approval
@tool(needs_approval=True)
async def cancel_order(order_id: int) -> str:
    return f"Cancelled order {order_id}"

##### Step-2

In [16]:
agent = Agent(
    name="File Assistant",
    instructions="Help the user manage files. Use the delete_file tool when asked.",
    tools=[cancel_order],
    model=llm_model,  # your model, as in the earlier examples
)

##### Step-3

In [26]:
# 3. Ask the human for a decision
def ask_human(tool_name: str, arguments: str) -> bool:
    answer = input(f"{tool_name}' with {arguments}? [y/N]: ")
    return answer.strip().lower() in ["y", "yes"]

##### Step-4

In [28]:
# 4. Run the agent. It pauses if a tool needs approval.
result = await Runner.run(agent, "Please cancel the order with ID 0235")

# 5. While the run is paused, there are pending approvals ("interruptions")
while result.interruptions:
    state = result.to_state()  # save the paused run

    for interruption in result.interruptions:
        if ask_human(interruption.name, interruption.arguments):
            state.approve(interruption)   # human said YES
        else:
            state.reject(interruption)    # human said NO

    # 6. Resume the SAME run with the decisions
    result = await Runner.run(agent, state)

print("\nFinal answer:", result.final_output)


Final answer: I am sorry, but I cannot fulfill this request. The tool execution was not approved.


In [29]:
from agents import ToolCallItem, ToolCallOutputItem

for item in result.new_items:
    if isinstance(item, ToolCallItem):
        print("TOOL CALLED:", item.raw_item.name)
        print("ARGUMENTS:  ", item.raw_item.arguments)
    elif isinstance(item, ToolCallOutputItem):
        print("TOOL OUTPUT:", item.output)

TOOL CALLED: cancel_order
ARGUMENTS:   {"order_id":235}
TOOL OUTPUT: Tool execution was not approved.
